In [ ]:
!pip install sleap==1.2.0a0
# !pip install imgaug==0.2.9

In [ ]:
import sleap
import numpy as np

sleap.versions()

SLEAP: 1.2.0a0
TensorFlow: 2.7.0
Numpy: 1.19.5
Python: 3.7.12
OS: Linux-5.4.104+-x86_64-with-Ubuntu-18.04-bionic


In [ ]:
from google.colab import files
uploads = files.upload()
labels_path = list(uploads.keys())[0]

Saving test_9_subsample_950.pkg.slp to test_9_subsample_950.pkg (1).slp


In [ ]:
labels_old = sleap.load_file(labels_path)
labels_old

ContextualVersionConflict: ignored

In [ ]:
centroid_skeleton = sleap.Skeleton()
centroid_skeleton.add_node("centroid")

nodes = labels_old.skeleton.node_names
centroid_nodes = ["spine1", "spine2", "spine3", "spine4", "spine5"]
centroid_inds = np.array([nodes.index(n) for n in centroid_nodes])

In [ ]:
# numpy -> Labels
lfs = []
for lf in labels_old:
    instances = []
    for instance in lf.instances:
        # Compute new centroid from reference nodes
        pose = instance.numpy()
        centroid_node_pts = pose[centroid_inds]
        if np.isnan(centroid_node_pts).all():
            # Skip if none of the reference nodes are visible.
            continue
        centroid = np.nanmedian(centroid_node_pts, axis=0, keepdims=True)

        instances.append(
            sleap.Instance.from_numpy(centroid, skeleton=centroid_skeleton, track=instance.track)
        )
    lfs.append(sleap.LabeledFrame(video=lf.video, frame_idx=lf.frame_idx, instances=instances))
labels_new = sleap.Labels(lfs)

labels_new

Labels(labeled_frames=2560, videos=1, skeletons=1, tracks=4)

In [ ]:
new_path = f"{labels_path[:-4]}.centroids.slp"
labels_new.save(new_path)

files.download(new_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>